# Notebook 9: PC1-adjusted GNAR

High-order GNAR comparison after removing the leading covariance principal component estimated from the training panel.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import gc
import time
import numpy as np
import pandas as pd
import pymc as pm

QUICK = quick_mode()

selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
K = int(selection["k"])

assert P == 38
assert selection["network"] == "geographic"
assert K == 2
assert int(selection["max_stage"]) == 2

d = load_data(network="geographic")
Y_full = d["Y_full"].to_numpy()
Y_train = d["Y_train"].to_numpy()
test_start = d["test_start"]
Y_hist = Y_full[:test_start]
Y_future_full = Y_full[test_start:]
test_dates_full = pd.to_datetime(d["test"].index)
N = d["N"]

EVAL_STEPS = min(3, len(Y_future_full)) if QUICK else len(Y_future_full)
test_dates = test_dates_full[:EVAL_STEPS]

W_geo = knn_sparsify(d["networks"]["geographic"], K)
W_uniform = (np.ones((N, N)) - np.eye(N)) / (N - 1)

assert int((compute_stage_weights(W_geo, 2)[1] > 0).sum()) > 0

kw = dict(NUTS_KW)
if QUICK:
    kw.update(draws=2, tune=2, chains=1, cores=1)
kw["idata_kwargs"] = {"log_likelihood": False}

## PC1 adjustment

The first covariance principal component is estimated from the training panel only. Its loading vector is held fixed when the evaluation-period series are adjusted.

In [2]:
# load_data() centres each country using the training-window mean, so covariance
# PCA is obtained directly from the training-centred panel.
U, singular_values, Vt = np.linalg.svd(Y_train, full_matrices=False)
pc1_loading = Vt[0].copy()

# Fix the otherwise arbitrary PCA sign for reproducible saved loadings.
if pc1_loading[np.argmax(np.abs(pc1_loading))] < 0:
    pc1_loading *= -1.0

pc1_variance_share = float(
    singular_values[0] ** 2 / np.sum(singular_values ** 2)
)

pc1_score_full = Y_full @ pc1_loading
Y_pc1_full = Y_full - np.outer(pc1_score_full, pc1_loading)

Y_pc1_train = Y_pc1_full[:test_start]
Y_pc1_hist = Y_pc1_full[:test_start]
Y_pc1_future_full = Y_pc1_full[test_start:]
Y_pc1_future = Y_pc1_future_full[:EVAL_STEPS]

assert np.max(np.abs(Y_pc1_full @ pc1_loading)) < 1e-10
assert Y_pc1_future.shape == (EVAL_STEPS, N)

pca_info = {
    "method": "covariance PCA",
    "training_months": int(len(Y_train)),
    "training_start": pd.Timestamp(d["train"].index[0]),
    "training_end": pd.Timestamp(d["train"].index[-1]),
    "pc1_variance_share": pc1_variance_share,
    "pc1_loadings": pc1_loading.tolist(),
}

print(f"PC1 variance share: {pc1_variance_share:.6f}")

PC1 variance share: 0.888292


## PC1-adjusted model comparison

All four models forecast the same PC1-adjusted response. Absolute scores are therefore interpreted only within this comparison.

In [3]:
MODEL_SPECS = {
    "ar": {
        "W": W_geo,
        "stages": [0] * P,
        "mode": "ar_only",
        "label": "AR(38)",
        "bundle": "nb9_pc1_ar_p38",
        "network": None,
        "k": None,
    },
    "uniform": {
        "W": W_uniform,
        "stages": [1] * P,
        "mode": "global_gnar",
        "label": "uniform",
        "bundle": "nb9_pc1_uniform_p38",
        "network": "uniform",
        "k": None,
    },
    "geo_stage1": {
        "W": W_geo,
        "stages": [1] * P,
        "mode": "global_gnar",
        "label": "geographic stage 1",
        "bundle": "nb9_pc1_geo_stage1_p38",
        "network": "geographic",
        "k": K,
    },
    "geo_stage2": {
        "W": W_geo,
        "stages": [2] * P,
        "mode": "global_gnar",
        "label": "geographic stage 2",
        "bundle": "nb9_pc1_geo_stage2_p38",
        "network": "geographic",
        "k": K,
    },
}

EXPECTED_COEFFICIENTS = {
    "ar": P,
    "uniform": 2 * P,
    "geo_stage1": 2 * P,
    "geo_stage2": 3 * P,
}

scores = {}
diagnostics = {}
monthly_crps = {}
forecast_bundles = {}
runtime_sec = {}
mean_parameter_counts = {}

def fit_pc1_model(key):
    """Fit and score one PC1-adjusted high-order specification."""
    spec = MODEL_SPECS[key]

    X, y, names = build_design(
        Y_pc1_train,
        spec["W"],
        p=P,
        stages=spec["stages"],
        mode=spec["mode"],
        h=1,
    )
    time_idx = build_time_index(Y_pc1_train, p=P, h=1)

    assert X.shape[0] == len(time_idx) * N
    assert len(names) == EXPECTED_COEFFICIENTS[key]

    model = build_gnar_sv(X, y, time_idx, names, N)

    with model:
        t0 = time.time()
        idata = pm.sample(**kw)
    elapsed = time.time() - t0

    diag = mcmc_diagnostics(idata)

    mean, var = forecast_online(
        idata,
        Y_pc1_hist,
        Y_pc1_future,
        spec["W"],
        p=P,
        stages=spec["stages"],
        mode=spec["mode"],
    )

    assert mean.shape == var.shape == Y_pc1_future.shape
    score = score_forecasts(
        Y_pc1_future, mean, var, test_dates
    )
    loss = crps_series(
        Y_pc1_future, mean, var
    )

    save_forecasts(
        spec["bundle"],
        Y_pc1_future,
        mean,
        var,
        test_dates,
        config=run_config(
            p=P,
            stages=spec["stages"],
            model=f"PC1-adjusted {spec['label']}",
            network=spec["network"],
            k=spec["k"],
            target="PC1-adjusted series",
            forecast_horizon=1,
            seed=NUTS_KW["random_seed"],
        ),
    )

    scores[key] = score
    diagnostics[key] = diag
    monthly_crps[key] = loss
    forecast_bundles[key] = spec["bundle"]
    runtime_sec[key] = float(elapsed)
    mean_parameter_counts[key] = len(names)

    print(
        f"{spec['label']}: {elapsed:.1f}s, CRPS={score['CRPS']:.6f}, "
        f"RMSE={score['RMSE']:.6f}, div={diag['divergences']}, "
        f"max R-hat={diag['max_rhat']:.5g}, "
        f"min bulk ESS={diag['min_ess_bulk']:.5g}, "
        f"min BFMI={diag['min_bfmi']:.5g}"
    )

    del idata, model, X, y
    gc.collect()

In [4]:
fit_pc1_model("ar")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 126 seconds.


AR(38): 132.5s, CRPS=0.222648, RMSE=0.440259, div=0, max R-hat=1, min bulk ESS=670, min BFMI=0.66327


In [5]:
fit_pc1_model("uniform")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 242 seconds.


uniform: 246.4s, CRPS=0.220591, RMSE=0.436306, div=0, max R-hat=1, min bulk ESS=1000, min BFMI=0.71367


In [6]:
fit_pc1_model("geo_stage1")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 238 seconds.


geographic stage 1: 242.6s, CRPS=0.220922, RMSE=0.436498, div=0, max R-hat=1, min bulk ESS=1300, min BFMI=0.68385


In [7]:
fit_pc1_model("geo_stage2")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 371 seconds.


geographic stage 2: 375.4s, CRPS=0.220171, RMSE=0.435007, div=0, max R-hat=1, min bulk ESS=1400, min BFMI=0.78368


## Matched PC1-adjusted comparisons

In [8]:
PAIR_SPECS = {
    "uniform_minus_ar": ("uniform", "ar"),
    "geo_stage1_minus_uniform": ("geo_stage1", "uniform"),
    "geo_stage2_minus_geo_stage1": ("geo_stage2", "geo_stage1"),
    "geo_stage2_minus_ar": ("geo_stage2", "ar"),
}

paired_monthly_losses = {}
comparisons = {}

for label, (first, second) in PAIR_SPECS.items():
    loss_first = monthly_crps[first]
    loss_second = monthly_crps[second]

    if loss_first.shape != loss_second.shape:
        raise ValueError(f"{label}: paired loss lengths differ.")

    diff = loss_first - loss_second
    dm_stat, dm_p = dm_test(
        loss_first, loss_second, h_dm=1
    )

    paired_monthly_losses[label] = diff
    comparisons[label] = {
        "sign_convention": "first-listed model CRPS minus second-listed model CRPS",
        "first_model": first,
        "second_model": second,
        "mean_loss_difference": float(diff.mean()),
        "dm_hln_stat": float(dm_stat),
        "dm_hln_p": float(dm_p),
    }

score_table = pd.DataFrame([
    {
        "model": key,
        "CRPS": scores[key]["CRPS"],
        "RMSE": scores[key]["RMSE"],
        "MAE": scores[key]["MAE"],
        "Coverage_95": scores[key]["Coverage_95"],
        "Width": scores[key]["Width"],
    }
    for key in MODEL_SPECS
])

print(score_table.round(6).to_string(index=False))
print()

for label, result in comparisons.items():
    print(
        f"{label}: {result['mean_loss_difference']:+.6f}, "
        f"DM p={result['dm_hln_p']:.5f}"
    )

     model     CRPS     RMSE      MAE  Coverage_95    Width
        ar 0.222648 0.440259 0.300179     0.907928 1.293265
   uniform 0.220591 0.436306 0.297525     0.910912 1.286644
geo_stage1 0.220922 0.436498 0.297446     0.910060 1.286272
geo_stage2 0.220171 0.435007 0.296351     0.908781 1.284758

uniform_minus_ar: -0.002057, DM p=0.00097
geo_stage1_minus_uniform: +0.000331, DM p=0.50190
geo_stage2_minus_geo_stage1: -0.000751, DM p=0.12043
geo_stage2_minus_ar: -0.002477, DM p=0.00660


## Save results

In [9]:
save_result("nb9_common_component", {
    "selection": selection,
    "pc1_adjusted": {
        "pca": pca_info,
        "scores": scores,
        "diagnostics": diagnostics,
        "runtime_sec": runtime_sec,
        "mean_parameter_counts": mean_parameter_counts,
        "forecast_bundles": forecast_bundles,
        "monthly_crps": {
            key: value.tolist()
            for key, value in monthly_crps.items()
        },
        "paired_monthly_losses": {
            key: value.tolist()
            for key, value in paired_monthly_losses.items()
        },
        "comparisons": comparisons,
    },
    "config": run_config(
        p=P,
        stages=[2] * P,
        network="geographic",
        k=K,
        max_stage=2,
        target="PC1-adjusted series",
        purpose="common_component_adjustment",
    ),
    "quick": QUICK,
})

print("saved nb9_common_component")

saved nb9_common_component
